# Experiment 2b: Data Cleaning and Transformation
## Automobile Dataset

| Field | Value |
|---|---|
| **Name** | Francis Reuben R |
| **USN** | 1RUA25CSE0168 |
| **Section** | G |
| **Course** | Data Science Laboratory |
| **Semester** | 3 |
| **Experiment** | 2b |


### Objective

Clean and transform the Automobile dataset so it is ready for analysis and modelling: handle missing values and duplicates, repair `horsepower`, recode `origin`, engineer features, encode categoricals, inspect outliers, scale numerical columns, and save `X`, `Y`, and the final preprocessed table.

### Dataset

UCI Auto MPG records. Columns: `name`, `mpg`, `cylinders`, `displacement`, `horsepower`, `weight`, `acceleration`, `model_year`, `origin`. `model_year` is stored as two digits (`70` means 1970).


## Setup


In [1]:
import pandas as pd
import numpy as np
from IPython.display import display
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler
from scipy import stats

STUDENT = {
    "name": "Francis Reuben R",
    "usn": "1RUA25CSE0168",
    "section": "G",
}
print(STUDENT)
pd.set_option("display.max_columns", 20)
pd.set_option("display.width", 120)


{'name': 'Francis Reuben R', 'usn': '1RUA25CSE0168', 'section': 'G'}


## Step 1: Load and Explore the Dataset


### 1. Load the dataset


In [2]:
df = pd.read_csv("Automobile.csv")
print("First 10 rows:")
display(df.head(10))
print("Shape (rows, columns):", df.shape)


First 10 rows:


,name,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,origin
0,chevrolet chevelle malibu,18.0,8.0,307.0,130.0,3504.0,12.0,70,usa
1,buick skylark 320,15.0,8.0,350.0,165.0,3693.0,11.5,70,usa
2,plymouth satellite,18.0,8.0,318.0,150.0,3436.0,11.0,70,usa
3,amc rebel sst,16.0,8.0,304.0,150.0,3433.0,12.0,70,usa
4,ford torino,17.0,NaN,302.0,140.0,3449.0,10.5,70,usa
5,ford galaxie 500,15.0,8.0,429.0,198.0,4341.0,10.0,70,usa
6,chevrolet impala,14.0,8.0,454.0,220.0,NaN,9.0,70,usa
7,plymouth fury iii,14.0,8.0,440.0,215.0,4312.0,8.5,70,usa
8,pontiac catalina,14.0,8.0,455.0,NaN,4425.0,10.0,70,usa
9,amc ambassador dpl,15.0,8.0,390.0,190.0,3850.0,8.5,70,usa


Shape (rows, columns): (398, 9)


### 2. Explore the dataset


In [3]:
print("Column names:")
print(list(df.columns))
print("\nData types:")
print(df.dtypes)
print("\nDescriptive statistics:")
display(df.describe(include="all"))

numeric_cols = df.select_dtypes(include=[np.number]).columns.tolist()
categorical_cols = df.select_dtypes(include=["object", "string", "category"]).columns.tolist()
print("Numerical columns:", numeric_cols)
print("Categorical columns:", categorical_cols)

print("\nUnique values in categorical columns:")
for col in categorical_cols:
    values = df[col].dropna().unique()
    print(f"  {col}: {len(values)} unique")
    if col != "name":
        print("   ", sorted(map(str, values)))
    else:
        print("   (car names omitted; high cardinality)")


Column names:
['name', 'mpg', 'cylinders', 'displacement', 'horsepower', 'weight', 'acceleration', 'model_year', 'origin']

Data types:
name                str
mpg             float64
cylinders       float64
displacement    float64
horsepower      float64
weight          float64
acceleration    float64
model_year        int64
origin              str
dtype: object

Descriptive statistics:


,name,mpg,cylinders,displacement,horsepower,weight,acceleration,model_year,origin
count,398,398.000000,395.000000,395.000000,386.000000,396.000000,395.000000,398.000000,398
unique,305,NaN,NaN,NaN,NaN,NaN,NaN,NaN,3
top,ford pinto,NaN,NaN,NaN,NaN,NaN,NaN,NaN,usa
freq,6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,249
mean,NaN,23.514573,5.445570,193.340506,104.316062,2965.025253,15.562278,76.010050,NaN
std,NaN,7.815984,1.696203,104.425993,38.086281,845.254458,2.750260,3.697627,NaN
min,NaN,9.000000,3.000000,68.000000,46.000000,1613.000000,8.000000,70.000000,NaN
25%,NaN,17.500000,4.000000,102.500000,75.250000,2222.250000,13.850000,73.000000,NaN
50%,NaN,23.000000,4.000000,146.000000,92.500000,2797.500000,15.500000,76.000000,NaN
75%,NaN,29.000000,8.000000,262.000000,125.000000,3581.750000,17.150000,79.000000,NaN


Numerical columns: ['mpg', 'cylinders', 'displacement', 'horsepower', 'weight', 'acceleration', 'model_year']
Categorical columns: ['name', 'origin']

Unique values in categorical columns:
  name: 305 unique
   (car names omitted; high cardinality)
  origin: 3 unique
    ['europe', 'japan', 'usa']


## Step 2: Data Cleaning


### 3. Check missing values


In [4]:
missing_count = df.isna().sum()
missing_pct = (df.isna().mean() * 100).round(2)
missing_table = pd.DataFrame({
    "missing_count": missing_count,
    "missing_percent": missing_pct,
})
display(missing_table)
print("Total missing cells:", int(missing_count.sum()))


,missing_count,missing_percent
name,0,0.00
mpg,0,0.00
cylinders,3,0.75
displacement,3,0.75
horsepower,12,3.02
weight,2,0.50
acceleration,3,0.75
model_year,0,0.00
origin,0,0.00


Total missing cells: 23


### 4. Handle missing values

Numerical gaps are filled with the **median** (robust to the long tail in horsepower and weight). Categorical gaps are filled with the **mode**. `name` has no missing values in this file.


In [5]:
work = df.copy()

num_missing = [c for c in numeric_cols if work[c].isna().any()]
cat_missing = [c for c in categorical_cols if work[c].isna().any()]
print("Numerical columns with NA:", num_missing)
print("Categorical columns with NA:", cat_missing)

for col in num_missing:
    median_value = work[col].median()
    work[col] = work[col].fillna(median_value)
    print(f"  filled {col} with median = {median_value}")

for col in cat_missing:
    mode_value = work[col].mode().iloc[0]
    work[col] = work[col].fillna(mode_value)
    print(f"  filled {col} with mode = {mode_value}")

print("\nMissing values after imputation:")
print(work.isna().sum())
assert work.isna().sum().sum() == 0, "Missing values still present"
print("No missing values remain.")


Numerical columns with NA: ['cylinders', 'displacement', 'horsepower', 'weight', 'acceleration']
Categorical columns with NA: []
  filled cylinders with median = 4.0
  filled displacement with median = 146.0
  filled horsepower with median = 92.5
  filled weight with median = 2797.5
  filled acceleration with median = 15.5

Missing values after imputation:
name            0
mpg             0
cylinders       0
displacement    0
horsepower      0
weight          0
acceleration    0
model_year      0
origin          0
dtype: int64
No missing values remain.


### 5. Remove duplicate records


In [6]:
n_dups = int(work.duplicated().sum())
print("Duplicate rows:", n_dups)
if n_dups:
    display(work[work.duplicated(keep=False)].head())
before = len(work)
work = work.drop_duplicates().reset_index(drop=True)
print("Rows before drop:", before)
print("Rows after drop:", len(work))
print("Duplicates remaining:", int(work.duplicated().sum()))


Duplicate rows: 0
Rows before drop: 398
Rows after drop: 398
Duplicates remaining: 0


### 6. Clean the `horsepower` column

Some Auto MPG copies store missing horsepower as the string `?`. This file already has NaN, but the same pipeline is applied: replace `?`, coerce to numeric, then impute.


In [7]:
print("horsepower dtype before:", work["horsepower"].dtype)
print("Non-numeric / '?' tokens:", int((work["horsepower"].astype(str).str.strip() == "?").sum()))
work["horsepower"] = work["horsepower"].replace("?", np.nan)
work["horsepower"] = pd.to_numeric(work["horsepower"], errors="coerce")
print("horsepower NaN after coerce:", int(work["horsepower"].isna().sum()))
if work["horsepower"].isna().any():
    hp_median = work["horsepower"].median()
    work["horsepower"] = work["horsepower"].fillna(hp_median)
    print("Filled remaining horsepower NA with median", hp_median)
print("horsepower dtype after:", work["horsepower"].dtype)
print("horsepower missing:", int(work["horsepower"].isna().sum()))


horsepower dtype before: float64
Non-numeric / '?' tokens: 0
horsepower NaN after coerce: 0
horsepower dtype after: float64
horsepower missing: 0


## Step 3: Data Transformation


### 7. Transform the `origin` column


In [8]:
print("Unique origin values before mapping:", sorted(work["origin"].astype(str).unique()))
origin_map = {
    1: "usa",
    2: "europe",
    3: "japan",
    "1": "usa",
    "2": "europe",
    "3": "japan",
    "usa": "usa",
    "europe": "europe",
    "japan": "japan",
}
work["origin"] = work["origin"].map(lambda v: origin_map.get(v, v))
print("Unique origin values after mapping:", sorted(work["origin"].astype(str).unique()))
print(work["origin"].value_counts())


Unique origin values before mapping: ['europe', 'japan', 'usa']
Unique origin values after mapping: ['europe', 'japan', 'usa']
origin
usa       249
japan      79
europe     70
Name: count, dtype: int64


### 8. Create `weight_kg`

`weight` is in pounds. Kilograms: `weight_kg = weight * 0.453592`.


In [9]:
work["weight_kg"] = work["weight"] * 0.453592
display(work[["name", "weight", "weight_kg"]].head())


,name,weight,weight_kg
0,chevrolet chevelle malibu,3504.0,1589.386368
1,buick skylark 320,3693.0,1675.115256
2,plymouth satellite,3436.0,1558.542112
3,amc rebel sst,3433.0,1557.181336
4,ford torino,3449.0,1564.438808


### 9. Create `mpg_category`

- mpg < 20: Low
- 20 to 29: Medium
- mpg >= 30: High


In [10]:
def mpg_category(mpg):
    if mpg < 20:
        return "Low"
    if mpg < 30:
        return "Medium"
    return "High"

work["mpg_category"] = work["mpg"].apply(mpg_category)
print(work["mpg_category"].value_counts())
display(work[["name", "mpg", "mpg_category"]].head())


mpg_category
Medium    155
Low       151
High       92
Name: count, dtype: int64


,name,mpg,mpg_category
0,chevrolet chevelle malibu,18.0,Low
1,buick skylark 320,15.0,Low
2,plymouth satellite,18.0,Low
3,amc rebel sst,16.0,Low
4,ford torino,17.0,Low


### 10. Create `vehicle_age`

`model_year` is two digits (`70` = 1970). Calendar year is `1900 + model_year`. Age in 2026:

`vehicle_age = 2026 - (1900 + model_year)`


In [11]:
work["vehicle_age"] = 2026 - (1900 + work["model_year"])
display(work[["name", "model_year", "vehicle_age"]].head())
print("Age range:", work["vehicle_age"].min(), "to", work["vehicle_age"].max(), "years")


,name,model_year,vehicle_age
0,chevrolet chevelle malibu,70,56
1,buick skylark 320,70,56
2,plymouth satellite,70,56
3,amc rebel sst,70,56
4,ford torino,70,56


Age range: 44 to 56 years


### 11. Rename columns


In [12]:
work = work.rename(columns={
    "mpg": "miles_per_gallon",
    "horsepower": "hp",
    "weight": "weight_lbs",
    "model_year": "year",
})
print("Columns after rename:")
print(list(work.columns))


Columns after rename:
['name', 'miles_per_gallon', 'cylinders', 'displacement', 'hp', 'weight_lbs', 'acceleration', 'year', 'origin', 'weight_kg', 'mpg_category', 'vehicle_age']


### 12. Filter the data

`year` is still two-digit, so manufactured after 1980 is `year > 80`.


In [13]:
print("miles_per_gallon > 30:", (work["miles_per_gallon"] > 30).sum(), "rows")
display(work.loc[work["miles_per_gallon"] > 30, ["name", "miles_per_gallon", "origin"]].head())

print("\nhp > 150:", (work["hp"] > 150).sum(), "rows")
display(work.loc[work["hp"] > 150, ["name", "hp", "cylinders"]].head())

print("\ncylinders >= 6:", (work["cylinders"] >= 6).sum(), "rows")
display(work.loc[work["cylinders"] >= 6, ["name", "cylinders", "hp"]].head())

print("\nManufactured after 1980 (year > 80):", (work["year"] > 80).sum(), "rows")
display(work.loc[work["year"] > 80, ["name", "year", "origin"]].head())

print("\norigin == usa:", (work["origin"] == "usa").sum(), "rows")
display(work.loc[work["origin"] == "usa", ["name", "origin", "year"]].head())


miles_per_gallon > 30: 85 rows


,name,miles_per_gallon,origin
53,toyota corolla 1200,31.0,japan
54,datsun 1200,35.0,japan
129,datsun b210,31.0,japan
131,toyota corolla 1200,32.0,japan
144,toyota corona,31.0,japan



hp > 150: 44 rows


,name,hp,cylinders
1,buick skylark 320,165.0,8.0
5,ford galaxie 500,198.0,8.0
6,chevrolet impala,220.0,8.0
7,plymouth fury iii,215.0,8.0
9,amc ambassador dpl,190.0,8.0



cylinders >= 6: 185 rows


,name,cylinders,hp
0,chevrolet chevelle malibu,8.0,130.0
1,buick skylark 320,8.0,165.0
2,plymouth satellite,8.0,150.0
3,amc rebel sst,8.0,150.0
5,ford galaxie 500,8.0,198.0



Manufactured after 1980 (year > 80): 60 rows


,name,year,origin
338,plymouth reliant,81,usa
339,buick skylark,81,usa
340,dodge aries wagon (sw),81,usa
341,chevrolet citation,81,usa
342,plymouth reliant,81,usa



origin == usa: 249 rows


,name,origin,year
0,chevrolet chevelle malibu,usa,70
1,buick skylark 320,usa,70
2,plymouth satellite,usa,70
3,amc rebel sst,usa,70
4,ford torino,usa,70


## Step 4: Encoding Categorical Data


### 13. Label encoding of `origin`


In [14]:
le = LabelEncoder()
work["origin_encoded"] = le.fit_transform(work["origin"])
mapping = {cls: int(code) for cls, code in zip(le.classes_, le.transform(le.classes_))}
print("Category-to-label mapping:", mapping)
display(work[["origin", "origin_encoded"]].drop_duplicates().sort_values("origin_encoded"))
display(work[["name", "origin", "origin_encoded"]].head())


Category-to-label mapping: {'europe': 0, 'japan': 1, 'usa': 2}


,origin,origin_encoded
19,europe,0
14,japan,1
0,usa,2


,name,origin,origin_encoded
0,chevrolet chevelle malibu,usa,2
1,buick skylark 320,usa,2
2,plymouth satellite,usa,2
3,amc rebel sst,usa,2
4,ford torino,usa,2


### 14. One-hot encoding of `origin`


In [15]:
origin_dummies = pd.get_dummies(work["origin"], prefix="origin", dtype=int)
display(origin_dummies.head())
print("Label encoding: one integer column, implied order", mapping)
print("One-hot encoding: one binary column per category, no implied order.")
print("One-hot is the better choice for origin. usa / europe / japan are nominal regions. Label codes would treat japan as 'greater than' usa, which is not a real order.")


,origin_europe,origin_japan,origin_usa
0,0,0,1
1,0,0,1
2,0,0,1
3,0,0,1
4,0,0,1


Label encoding: one integer column, implied order {'europe': 0, 'japan': 1, 'usa': 2}
One-hot encoding: one binary column per category, no implied order.
One-hot is the better choice for origin. usa / europe / japan are nominal regions. Label codes would treat japan as 'greater than' usa, which is not a real order.


## Step 5: Outlier Detection


### 15. Identify outliers using Z-scores (|Z| > 3)


In [16]:
z_cols = ["miles_per_gallon", "cylinders", "displacement", "hp", "weight_lbs", "acceleration", "year"]
z_abs = np.abs(stats.zscore(work[z_cols], nan_policy="omit"))
z_df = pd.DataFrame(z_abs, columns=z_cols, index=work.index)
outlier_flags = z_df > 3
print("Outlier count by column:")
print(outlier_flags.sum())
print("Rows with at least one |Z| > 3:", int(outlier_flags.any(axis=1).sum()))
outlier_rows = work.loc[outlier_flags.any(axis=1), ["name"] + z_cols]
display(outlier_rows)
print("Decision: retain. The flagged rows are known high-output V8s (Impala, Electra, Grand Prix) and two slow-acceleration economy cars. They are real vehicles, not sensor errors, so they stay in the table.")


Outlier count by column:
miles_per_gallon    0
cylinders           0
displacement        0
hp                  4
weight_lbs          0
acceleration        2
year                0
dtype: int64
Rows with at least one |Z| > 3: 6


,name,miles_per_gallon,cylinders,displacement,hp,weight_lbs,acceleration,year
6,chevrolet impala,14.0,8.0,454.0,220.0,2797.5,9.0,70
13,buick estate wagon (sw),14.0,8.0,455.0,225.0,3086.0,10.0,70
95,buick electra 225 custom,12.0,8.0,455.0,225.0,4951.0,11.0,73
116,pontiac grand prix,16.0,8.0,400.0,230.0,4278.0,9.5,73
299,peugeot 504,27.2,4.0,141.0,71.0,3190.0,24.8,79
394,vw pickup,44.0,4.0,97.0,52.0,2130.0,24.6,82


Decision: retain. The flagged rows are known high-output V8s (Impala, Electra, Grand Prix) and two slow-acceleration economy cars. They are real vehicles, not sensor errors, so they stay in the table.


## Step 6: Feature Scaling


### 16. Standardization (StandardScaler)


In [17]:
scale_cols = ["miles_per_gallon", "cylinders", "displacement", "hp", "weight_lbs", "acceleration", "year", "weight_kg", "vehicle_age"]
df_std = work.copy()
scaler_std = StandardScaler()
df_std[scale_cols] = scaler_std.fit_transform(work[scale_cols])
print("Standardized head:")
display(df_std[scale_cols].head())
print("Means (expect ~0):")
print(df_std[scale_cols].mean().round(10))
print("\nStandard deviations (expect ~1):")
print(df_std[scale_cols].std(ddof=0).round(10))


Standardized head:


,miles_per_gallon,cylinders,displacement,hp,weight_lbs,acceleration,year,weight_kg,vehicle_age
0,-0.706439,1.515897,1.096516,0.694154,0.641001,-1.301636,-1.627426,0.641001,1.627426
1,-1.090751,1.515897,1.510055,1.627150,0.865428,-1.484357,-1.627426,0.865428,1.627426
2,-0.706439,1.515897,1.202305,1.227295,0.560255,-1.667078,-1.627426,0.560255,1.627426
3,-0.962647,1.515897,1.067664,1.227295,0.556693,-1.301636,-1.627426,0.556693,1.627426
4,-0.834543,-0.847774,1.048430,0.960725,0.575692,-1.849799,-1.627426,0.575692,1.627426


Means (expect ~0):
miles_per_gallon    0.0
cylinders          -0.0
displacement       -0.0
hp                  0.0
weight_lbs         -0.0
acceleration       -0.0
year               -0.0
weight_kg           0.0
vehicle_age        -0.0
dtype: float64

Standard deviations (expect ~1):
miles_per_gallon    1.0
cylinders           1.0
displacement        1.0
hp                  1.0
weight_lbs          1.0
acceleration        1.0
year                1.0
weight_kg           1.0
vehicle_age         1.0
dtype: float64


## Step 7: Normalization


### 17. Min-max scaling to [0, 1]


In [18]:
df_norm = work.copy()
scaler_mm = MinMaxScaler()
df_norm[scale_cols] = scaler_mm.fit_transform(work[scale_cols])
print("Normalized head:")
display(df_norm[scale_cols].head())
print("Min (expect 0):")
print(df_norm[scale_cols].min())
print("\nMax (expect 1):")
print(df_norm[scale_cols].max())
print("\nStandardization (z-score) is appropriate when features are roughly Gaussian or when a model cares about variance (linear regression, logistic regression, SVM, k-means, PCA).")
print("Min-max normalization is appropriate when a bounded [0, 1] range is required (neural nets, k-NN) and when outliers have already been treated, because min and max are sensitive to extremes.")
print("For this automobile table, StandardScaler is the better default: horsepower and weight have long right tails, and later labs are likely to use linear or distance-based models.")


Normalized head:


,miles_per_gallon,cylinders,displacement,hp,weight_lbs,acceleration,year,weight_kg,vehicle_age
0,0.239362,1.0,0.617571,0.456522,0.536150,0.238095,0.0,0.536150,1.0
1,0.159574,1.0,0.728682,0.646739,0.589736,0.208333,0.0,0.589736,1.0
2,0.239362,1.0,0.645995,0.565217,0.516870,0.178571,0.0,0.516870,1.0
3,0.186170,1.0,0.609819,0.565217,0.516019,0.238095,0.0,0.516019,1.0
4,0.212766,0.2,0.604651,0.510870,0.520556,0.148810,0.0,0.520556,1.0


Min (expect 0):
miles_per_gallon    0.0
cylinders           0.0
displacement        0.0
hp                  0.0
weight_lbs          0.0
acceleration        0.0
year                0.0
weight_kg           0.0
vehicle_age         0.0
dtype: float64

Max (expect 1):
miles_per_gallon    1.0
cylinders           1.0
displacement        1.0
hp                  1.0
weight_lbs          1.0
acceleration        1.0
year                1.0
weight_kg           1.0
vehicle_age         1.0
dtype: float64

Standardization (z-score) is appropriate when features are roughly Gaussian or when a model cares about variance (linear regression, logistic regression, SVM, k-means, PCA).
Min-max normalization is appropriate when a bounded [0, 1] range is required (neural nets, k-NN) and when outliers have already been treated, because min and max are sensitive to extremes.
For this automobile table, StandardScaler is the better default: horsepower and weight have long right tails, and later labs are likely to us

## Step 8: Create Features and Target


### 18. Create X and Y

Target `Y` is `miles_per_gallon`. `X` uses the physical predictors. `name` is dropped (305 unique strings). `mpg_category` is dropped (derived from the target). `weight_kg` is dropped because it is a linear transform of `weight_lbs`. `vehicle_age` is dropped because it is a linear transform of `year`. Origin enters as one-hot columns (no false order).


In [19]:
X = pd.concat([
    work[["cylinders", "displacement", "hp", "weight_lbs", "acceleration", "year"]],
    origin_dummies,
], axis=1)
Y = work["miles_per_gallon"].rename("miles_per_gallon")
print("X shape:", X.shape)
print("Y shape:", Y.shape)
print("X columns:", list(X.columns))
xy = X.copy()
xy["miles_per_gallon"] = Y
xy.to_csv("automobile_X_Y.csv", index=False)
xy_loaded = pd.read_csv("automobile_X_Y.csv")
print("\nReloaded automobile_X_Y.csv, first 5 rows:")
display(xy_loaded.head())


X shape: (398, 9)
Y shape: (398,)
X columns: ['cylinders', 'displacement', 'hp', 'weight_lbs', 'acceleration', 'year', 'origin_europe', 'origin_japan', 'origin_usa']

Reloaded automobile_X_Y.csv, first 5 rows:


,cylinders,displacement,hp,weight_lbs,acceleration,year,origin_europe,origin_japan,origin_usa,miles_per_gallon
0,8.0,307.0,130.0,3504.0,12.0,70,0,0,1,18.0
1,8.0,350.0,165.0,3693.0,11.5,70,0,0,1,15.0
2,8.0,318.0,150.0,3436.0,11.0,70,0,0,1,18.0
3,8.0,304.0,150.0,3433.0,12.0,70,0,0,1,16.0
4,4.0,302.0,140.0,3449.0,10.5,70,0,0,1,17.0


## Step 9: Save the Final Dataset


### 19. Save the preprocessed dataset


In [20]:
final = pd.concat([
    work[["name", "miles_per_gallon", "cylinders", "displacement", "hp", "weight_lbs", "weight_kg",
          "acceleration", "year", "vehicle_age", "mpg_category", "origin", "origin_encoded"]],
    origin_dummies,
], axis=1)
print("Final dataset shape:", final.shape)
display(final.head())
print("Missing values in final dataset:")
print(final.isna().sum())
assert final.isna().sum().sum() == 0
final.to_csv("automobile_preprocessed.csv", index=False)
print("Saved automobile_preprocessed.csv")


Final dataset shape: (398, 16)


,name,miles_per_gallon,cylinders,displacement,hp,weight_lbs,weight_kg,acceleration,year,vehicle_age,mpg_category,origin,origin_encoded,origin_europe,origin_japan,origin_usa
0,chevrolet chevelle malibu,18.0,8.0,307.0,130.0,3504.0,1589.386368,12.0,70,56,Low,usa,2,0,0,1
1,buick skylark 320,15.0,8.0,350.0,165.0,3693.0,1675.115256,11.5,70,56,Low,usa,2,0,0,1
2,plymouth satellite,18.0,8.0,318.0,150.0,3436.0,1558.542112,11.0,70,56,Low,usa,2,0,0,1
3,amc rebel sst,16.0,8.0,304.0,150.0,3433.0,1557.181336,12.0,70,56,Low,usa,2,0,0,1
4,ford torino,17.0,4.0,302.0,140.0,3449.0,1564.438808,10.5,70,56,Low,usa,2,0,0,1


Missing values in final dataset:
name                0
miles_per_gallon    0
cylinders           0
displacement        0
hp                  0
weight_lbs          0
weight_kg           0
acceleration        0
year                0
vehicle_age         0
mpg_category        0
origin              0
origin_encoded      0
origin_europe       0
origin_japan        0
origin_usa          0
dtype: int64
Saved automobile_preprocessed.csv


### Result

Missing numerical values were median-imputed, `horsepower` was coerced to numeric, origin labels were standardised, and derived columns (`weight_kg`, `mpg_category`, `vehicle_age`, `origin_encoded`, one-hot origin) were added. Six Z-score outliers were inspected and retained. `X` has 9 columns and `Y` is `miles_per_gallon`. Files: `automobile_X_Y.csv`, `automobile_preprocessed.csv`.

### Conclusion

The Automobile table was cleaned and transformed into analysis-ready features. One-hot encoding is the correct treatment of `origin`. Standardization is the preferred scaler for this dataset. Target leakage columns were kept out of `X`.
